## 3 - Noise reduction

[MNE's OPM tutorial](https://mne.tools/stable/auto_examples/preprocessing/opm_data.html) uses two approaches to
remove environmental magnetic interference: regressing out the reference sensors, and Homogeneous Field Correction
(HFC). The FLUX pipeline only uses the latter. In this notebook, we compare the two methods:

- Step 3: preparing the spectra (settings, and the spectrum before any correction)
- Step 4a: reference regression
- Step 4b: HFC
- Step 5: what HFC removed, checked against the reference sensors
- Step 6: both methods side by side, and combined

## Step 0: Load the Python packages


In [ ]:
# --- Standard Python tools (come with Python) ---
from pathlib import Path        # build and check file/folder paths

# --- Scientific Python tools ---
import numpy as np              # numbers and arrays (lists of numbers you can do math on)
import pandas as pd             # tables
import matplotlib.pyplot as plt # plotting
from IPython.display import display   # shows a table nicely in the notebook

# --- MNE-Python: the MEG analysis package ---
import mne

# --- MNE-BIDS: reads BIDS files together with their sidecar files (Step 2) ---
import mne_bids

# Only show MNE's warnings and errors (not every info message)
mne.set_log_level("WARNING")

# Draw plots inside the notebook
%matplotlib inline

print("MNE version:", mne.__version__)
print("MNE-BIDS version:", mne_bids.__version__)

## Step 1: Settings ✏️ (edit this cell)

Everything you normally need to change is in the cell below.

In [ ]:
# ---------------------------------------------------------------------------
# WHICH RECORDING
# ---------------------------------------------------------------------------
BIDS_ROOT = Path("/jukebox/[yourpath]/auditory-oddball/bids")  # top folder of your BIDS dataset
SUBJECT = "01"           # the part after "sub-"  (as text, in quotes)
SESSION = "01"           # the part after "ses-"; use None (no quotes) if you have no sessions
TASK = "oddballftcued"   # the part after "task-"
RUN = None               # the part after "run-", e.g. "01"; use None if there is no run label
FILTERED_DERIV = "filtered"  # the derivatives folder notebook 02 saved the filtered, downsampled recording to

# ---------------------------------------------------------------------------
# SAVING
# ---------------------------------------------------------------------------
OVERWRITE = False            # True: replace the saved recording from an earlier run of this notebook

## Step 2: Load the filtered, downsampled recording

This cell loads the recording that notebook 02 saved in `derivatives/filtered/`. It works out the file's path from
your settings, checks it exists, and reads it. If something is wrong it stops with a message that says what to fix.

Notebook 02 saved everything the earlier notebooks found next to the recording, so this is the only thing to load
(nothing from `derivatives/sensorqc/` or `derivatives/eyesync/`):

- the **bad channels** from notebook 01, with their reasons (from `_channels.tsv`);
- the **task events**, the **eye-tracking events** (`ET_...`) and the **breaks** (`BAD_break`), as annotations
  (from `_events.tsv`);
- the **eye-tracker channels**, in the same file as the MEG channels.

The cell ends with a summary of what was loaded, so you can check it before going on.

In [ ]:
BIDS_ROOT = Path(BIDS_ROOT)

if "[yourpath]" in str(BIDS_ROOT):
    raise ValueError("BIDS_ROOT still contains the [yourpath] placeholder. Set it in Step 1.")

# ---- The BIDS-style name of this recording, e.g. sub-01_ses-01_task-oddball ------------------
sub_ses = [f"sub-{SUBJECT}"] + ([f"ses-{SESSION}"] if SESSION else [])   # folder levels
name_parts = sub_ses + [f"task-{TASK}"] + ([f"run-{RUN}"] if RUN else [])
base = "_".join(name_parts)        # the start of every file name for this recording
sub_dir = Path(*sub_ses)           # e.g. sub-01/ses-01

# ---- The filtered recording from notebook 02 --------------------------------------------------
filtered_root = BIDS_ROOT / "derivatives" / FILTERED_DERIV
# BIDSPath describes a file by its BIDS labels. Notebook 02 labels the filtered recording "proc-filt".
bids_path = mne_bids.BIDSPath(root=filtered_root, subject=SUBJECT, session=SESSION, task=TASK, run=RUN,
                              processing="filt", datatype="meg", suffix="meg", extension=".fif")
# A recording over 2 GB is saved in parts (..._split-01_meg.fif, ...): open the first, MNE finds the others.
if not bids_path.fpath.is_file() and bids_path.copy().update(split="01").fpath.is_file():
    bids_path.update(split="01")
meg_file = bids_path.fpath
meg_dir = meg_file.parent

if not meg_file.is_file():
    found = sorted(meg_dir.glob("*_meg.fif")) if meg_dir.is_dir() else []
    raise FileNotFoundError(f"Filtered recording not found:\n    {meg_file}\nHas notebook 02 been run for this "
                            "session, up to its Step 4 (saving)? FIF files in that folder: "
                            + (", ".join(p.name for p in found) or "none"))

# ---- Read it --------------------------------------------------------------------------------
# read_raw_bids reads the FIF file together with the sidecar files notebook 02 wrote next to it:
#   _channels.tsv -> the channel types and the bad channels (notebook 01's list)
#   _events.tsv   -> the task events, eye-tracking events and breaks, as annotations
# The warning "participants.tsv file not found" is expected and harmless (see notebook 02, Step 2).
raw = mne_bids.read_raw_bids(bids_path)
raw.load_data()   # read the data themselves into memory (the steps below need them)

print("Loaded:", meg_file)
if len(raw.filenames) > 1:
    print(f"(the recording is split into {len(raw.filenames)} files; MNE read them all)")

# read_raw_bids loses the coil type of the eye-tracker channels (see notebook 01, Step 2b);
# the FIF file's own header still has it, so copy it back. No file is changed.
header = mne.io.read_info(meg_file, verbose=False)     # header only: quick, no data
for ch_in_file in header["chs"]:
    ch = raw.info["chs"][raw.ch_names.index(ch_in_file["ch_name"])]
    if ch["kind"] == ch_in_file["kind"]:
        ch["coil_type"] = ch_in_file["coil_type"]
types = dict(zip(raw.ch_names, raw.get_channel_types()))   # errors if the repair didn't work

# ---- What was loaded --------------------------------------------------------------------------
print(f"\nSampling rate: {raw.info['sfreq']:g} Hz   Duration: {raw.times[-1] / 60:.1f} min   "
      f"Filter: {raw.info['highpass']:g}-{raw.info['lowpass']:g} Hz")
print("Channels:", pd.Series(list(types.values())).value_counts().to_dict())
print("Eye-tracker channels:", [n for n, t in types.items() if t in ("eyegaze", "pupil")])
print(f"Bad channels ({len(raw.info['bads'])}):", raw.info["bads"] or "none")

## Step 3: Prepare the spectra 

This cell prepares what the following steps need to compare the spectra. It doesn't plot anything:

- `psd_settings`: the settings every spectrum in this notebook uses, so they can be compared directly. Welch's
  method with windows of `n_fft` samples (1500 samples at 600 Hz = 2.5 s, so one value every 0.4 Hz), up to
  120 Hz. The bad channels and the breaks are left out.
- `raw_PSD`: the power spectral density (PSD) of the good head sensors **before any correction**. It is computed
  once here, and shown as the "no correction" curve in the plots of Steps 4a, 4b, 5 and 6.
- `mean_db()`: averages the power over the channels and converts it to dB relative to 1 fT²/Hz; the plots use it.

The cell prints how many channels the spectrum includes, to confirm the bad channels were left out.

In [ ]:
n_fft = 1500     # samples per window: 2.5 s at 600 Hz -> 0.4 Hz frequency resolution
psd_settings = dict(method="welch", fmin=0.1, fmax=120, picks="mag", exclude="bads",
                    n_fft=n_fft, n_overlap=int(n_fft / 2))

raw_PSD = raw.compute_psd(**psd_settings)    # units: T²/Hz

def mean_db(spectrum):
    # mean power over the channels (T²/Hz) -> dB relative to 1 fT²/Hz 
    return 10 * np.log10(spectrum.get_data().mean(axis=0) * 1e30)

## Step 4a: Reference regression

The reference sensors sit away from the head, so they only measure the environment. **Regressing them out**
removes from the head sensors what they have in common with the reference sensors.

The reference sensors are first low-pass filtered at `REF_LOWPASS_HZ` (5 Hz), as in MNE's tutorial: only the slow
environmental fields are regressed out, because above that the reference sensors mostly carry their own sensor
noise, which the regression would add to the head sensors.

The regression is fitted on the whole recording, the breaks included. The regressed recording `raw_reg` is kept
until Step 6, which adds HFC on top of it and then deletes it (each copy of the recording takes a few GB).

In [ ]:
REF_LOWPASS_HZ = 5      # the reference sensors are low-pass filtered at this frequency before the regression

# ---- Reference regression, on a copy of raw ----------------------------------------------------------------
raw_reg = raw.copy()
raw_reg.filter(None, REF_LOWPASS_HZ, picks="ref_meg")       # only the reference sensors are filtered
regress = mne.preprocessing.EOGRegression(picks="mag", exclude="bads", picks_artifact="ref_meg") # exclude bad channels  
regress.fit(raw_reg)                                         # one weight per head sensor and reference sensor
regress.apply(raw_reg, copy=False)                           # subtract the predicted environmental field
print(f"Regressed the reference sensors out of {regress.coef_.shape[0]} good head sensors")

raw_PSDreg = raw_reg.compute_psd(**psd_settings)

# ---- Spectrum before and after the regression --------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4.5))
for spectrum, label, color in [(raw_PSD, "no correction", "0.5"), (raw_PSDreg, "reference regression", "C1")]:
    ax.plot(spectrum.freqs, mean_db(spectrum), color=color, label=label)
ax.axvline(REF_LOWPASS_HZ, color="C1", ls=":", lw=1, label=f"low-pass of the references ({REF_LOWPASS_HZ} Hz)")
ax.set(xscale="log", xlabel="Frequency (Hz)", ylabel="Power (dB re 1 fT²/Hz), mean over good head sensors",
       title=f"{base}: before and after reference regression")
ax.legend()
plt.show()

## Step 4b: Homogeneous Field Correction (HFC)

Regression of a reference channel assumes that the relationship between the references and a given sensor on the head is constant. However this becomes less accurate when the reference is not moving but the subject is. An alternative method, Homogeneous Field Correction (HFC) only requires that the sensors on the helmet are stationary relative to each other It models the interference
from the head sensors themselves, using each sensor's position and orientation.

Both Flux and MNE tutorials set with `order=2`. For more information, see Tierney et al., 2021, NeuroImage, doi:10.1016/j.neuroimage.2021.118484.

In [ ]:
raw_hfc=raw.copy()
projs = mne.preprocessing.compute_proj_hfc(raw_hfc.info, order=2, exclude="bads")# Exclude bad channels
raw_hfc.add_proj(projs).apply_proj(verbose="error")
print(f"{len(projs)} HFC projectors, built from {len(projs[0]['data']['col_names'])} channels")


### Time course before and after HFC

Note the large artefacts around 1000 s and 2000 s. They are probably due to head movements, as the participant
is stretching and repositioning.

In [ ]:
picks = mne.pick_types(raw.info, meg=True, exclude="bads") # Plot the good MEG channels only
amp_scale = 1e12            # T -> pT
step = 500                  # plot every 500th sample, to keep it light

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, rec, title in [(axes[0], raw, "Before HFC"), (axes[1], raw_hfc, "After HFC")]:
    data, times = rec.get_data(picks=picks, return_times=True)
    data = data[:, ::step] * amp_scale
    ax.plot(times[::step], (data - data.mean(axis=1, keepdims=True)).T, lw=0.5) # subtract channel mean to center to zero
    ax.set(title=title, xlabel="Time (s)")
axes[0].set_ylabel("Amplitude (pT), each channel minus its mean")
axes[0].set_ylim(-500, 500)
plt.show()


### Spectra before and after HFC

The spectrum after HFC uses the same settings as the one before (Step 3). The first plot shows every channel; the
second, their mean.

In [ ]:
raw_PSDc = raw_hfc.compute_psd(**psd_settings)     # after HFC; raw_PSD (before) is from Step 3

# Every channel, in dB relative to 1 fT²/Hz, for the plots below
freqs, freqsc = raw_PSD.freqs, raw_PSDc.freqs
psd_db = 10 * np.log10(raw_PSD.get_data() * 1e30)
psd_dbc = 10 * np.log10(raw_PSDc.get_data() * 1e30)

In [ ]:
# plot PSD before and after HFC for each channel

plt.figure(figsize=(20, 6))

plt.subplot(1,2,1)
plt.plot(freqs,psd_db.T);plt.ylim(20, 80);plt.xlabel('Freq [Hz]');plt.ylabel('Power [fT^2/Hz]');plt.title('NO HFC');

plt.subplot(1,2,2)
plt.plot(freqsc,psd_dbc.T);plt.ylim(20, 80);plt.xlabel('Freq [Hz]');plt.ylabel('Power [fT^2/Hz]');plt.title('HFC');

In [ ]:
# plot average PSD before and after HFC
plt.plot(freqsc,np.mean(psd_dbc.T,axis=1), label='HFC')
plt.plot(freqs,np.mean(psd_db.T,axis=1),label='no HFC')
plt.legend();plt.ylim(20, 100);plt.xlabel('Freq [Hz]');
plt.ylabel('Power (dB) [fT^2/Hz]');plt.title('mean power over OPM sensors');

## Step 5: Check: what did HFC remove? Comparison with the reference sensors

HFC removes fields that are the same across the whole helmet, or change evenly across it. These come from the
environment, but a reduction in the spectrum could in principle also be brain activity. The **reference sensors**
help to tell the two apart: they sit away from the head, so they only measure the environment. HFC doesn't change
them (it only uses the head sensors).

- A peak or a rise that the reference sensors **also** show comes from the environment. If HFC reduces it on the
  head sensors, that's interference removed (for example the narrow peak at 25 Hz in sub-001).
- A peak the reference sensors **don't** show (for example alpha around 10 Hz) is likely brain activity, and it
  should still be there after HFC.

The left plot shows the whole spectrum (log frequency axis), the right plot zooms in on 8-40 Hz (alpha and beta).
The three curves have different absolute levels: compare their **shapes**, not their heights.

In [ ]:
# Reference sensors: same settings as the head-sensor spectra. HFC doesn't change them, so raw is enough.
raw_PSDref = raw.compute_psd(**dict(psd_settings, picks="ref_meg"))

curves = [(raw_PSD, "head sensors, before HFC", "0.5"),
          (raw_PSDc, "head sensors, after HFC", "C0"),
          (raw_PSDref, "reference sensors", "C3")]

fig, (ax_all, ax_zoom) = plt.subplots(1, 2, figsize=(14, 5))
for spectrum, label, color in curves:
    f, db = spectrum.freqs, mean_db(spectrum)
    ax_all.plot(f, db, color=color, label=label)
    zoom = (f >= 2) & (f <= 30)
    ax_zoom.plot(f[zoom], db[zoom], color=color, label=label)
ax_all.set(xscale="log", title="Whole spectrum (log frequency axis)")
ax_zoom.set(title="Zoom on 2-30 Hz (linear axis)")
for ax in (ax_all, ax_zoom):
    ax.set(xlabel="Frequency (Hz)", ylabel="Power (dB re 1 fT²/Hz), mean over channels")
ax_zoom.legend()
fig.suptitle(f"{base}: head sensors before and after HFC, and the reference sensors")
fig.tight_layout()
plt.show()

## Step 6: Comparison: reference regression, HFC, and both

The two methods use different information:

- **HFC** uses only the head sensors: it removes what is uniform (or changes evenly) across the helmet, at all
  frequencies.
- **Reference regression** uses only the reference sensors: it removes what the head sensors have in common with
  them, here only below 5 Hz.

This cell adds HFC on top of the reference regression from Step 4a, and compares the
spectra of the good head sensors for: no correction, HFC, reference regression, and both. The spectra of the first
three come from Steps 3, 4a and 4b. The regressed recording is deleted at the end, to free memory.

In [ ]:
# ---- Both: HFC on top of the reference regression from Step 4a (in place, to save memory) ---------------------
raw_reg.add_proj(mne.preprocessing.compute_proj_hfc(raw_reg.info, order=2, exclude="bads"))
raw_reg.apply_proj(verbose="error")
raw_PSDboth = raw_reg.compute_psd(**psd_settings)
del raw_reg                                                  # only the spectra are needed from here on

# ---- Spectra ---------------------------------------------------------------------------------------------
curves = [(raw_PSD, "no correction", "0.5"),
          (raw_PSDc, "HFC", "C0"),
          (raw_PSDreg, "reference regression", "C1"),
          (raw_PSDboth, "reference regression + HFC", "C2")]
fig, (ax_all, ax_zoom) = plt.subplots(1, 2, figsize=(14, 5))
for spectrum, label, color in curves:
    f, db = spectrum.freqs, mean_db(spectrum)
    ax_all.plot(f, db, color=color, label=label)
    zoom = f <= 40
    ax_zoom.plot(f[zoom], db[zoom], color=color, label=label)
ax_all.set(xscale="log", title="Whole spectrum (log frequency axis)")
ax_zoom.set(title="0-40 Hz (linear axis)")
for ax in (ax_all, ax_zoom):
    ax.axvline(REF_LOWPASS_HZ, color="C1", ls=":", lw=1)
    ax.set(xlabel="Frequency (Hz)", ylabel="Power (dB re 1 fT²/Hz), mean over good head sensors")
ax_zoom.legend()
fig.suptitle(f"{base}: HFC and reference regression (dotted line: the {REF_LOWPASS_HZ} Hz low-pass of the references)")
fig.tight_layout()
plt.show()

Note: Line noise is still prominent at 60 Hz and harmonics.